In [ ]:
# Search for pages
results = confluence.cql('text ~ "keyword" AND type = page')
print(f"Found {len(results)} pages")
for page in results:
    print(f"  - {page['title']} (ID: {page['id']})")

In [10]:
import os
from datetime import datetime
import re

def generate_confluence_report(confluence, output_file="confluence_pages_report.md", search_query='type = page', limit=None):
    """
    Search Confluence and generate a consolidated markdown report of all pages found.
    
    Parameters:
    - confluence: Confluence connection object
    - output_file: Output markdown filename
    - search_query: CQL query (default: all pages)
    - limit: Maximum number of pages to include (None = all)
    """
    
    print(f"Searching Confluence for pages...")
    try:
        # Search for pages
        response = confluence.cql(search_query, limit=limit, start=0)
        
        # Handle the response - it should be a dict with 'results' key
        if isinstance(response, dict):
            results = response.get('results', [])
        elif isinstance(response, list):
            results = response
        else:
            print(f"Unexpected response type: {type(response)}")
            return
        
        if not results:
            print("No pages found.")
            return
        
        print(f"Found {len(results)} pages, generating report...")
        
        # Start building markdown content
        md_content = f"""# Confluence Pages Report
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Summary
- **Total Pages Found:** {len(results)}
- **Search Query:** {search_query}

## Table of Contents
"""
        
        # Add TOC
        for idx, result in enumerate(results, 1):
            title = result.get('title', 'Untitled')
            safe_title = title.replace('[', '').replace(']', '')
            md_content += f"\n{idx}. [{safe_title}](#{idx}-{safe_title.lower().replace(' ', '-')})"
        
        # Add detailed pages section
        md_content += "\n\n---\n\n## Pages Details\n"
        
        for idx, result in enumerate(results, 1):
            title = result.get('title', 'Untitled')
            
            # CQL response nests the actual content object - extract page info from there
            content = result.get('content', {})
            page_id = content.get('id', 'N/A')
            
            space_key = 'N/A'
            # Try to extract space key - it might be in the URL
            url = result.get('url', '')
            if '/spaces/' in url:
                # URL format: /spaces/~<space_key>/pages/<page_id>/<title>
                space_part = url.split('/spaces/')[1].split('/')[0]
                space_key = space_part if space_part else 'N/A'
            
            # Get content preview from excerpt first (already available)
            content_preview = result.get('excerpt', 'Unable to retrieve content preview')
            content_preview = content_preview.strip()
            
            # If excerpt is too short, try to fetch full content
            if not content_preview or len(content_preview) < 50:
                try:
                    if page_id and page_id != 'N/A':
                        print(f"  Fetching details for: {title} (ID: {page_id})")
                        full_page = confluence.get_page_by_id(page_id, expand="body.storage")
                        
                        body_content = full_page.get('body', {}).get('storage', {}).get('value', '')
                        if body_content:
                            # Remove HTML tags properly
                            text = re.sub('<[^<]+?>', '', body_content)
                            text = text.strip()
                            # Replace multiple spaces/newlines
                            text = re.sub(r'\s+', ' ', text)
                            content_preview = text[:500]
                            if len(text) > 500:
                                content_preview += "..."
                except Exception as e:
                    print(f"    Warning: Could not fetch full content: {str(e)}")
            
            # Build URL
            if url.startswith('/'):
                full_url = f"{confluence.url}{url}"
            else:
                full_url = url if url else f"{confluence.url}/pages/viewpage.action?pageId={page_id}"
            
            md_content += f"""
### {idx}. {title}
- **Page ID:** {page_id}
- **Space Key:** {space_key}
- **URL:** {full_url}
- **Content Preview:**
  {content_preview}

---
"""
        
        # Write to file
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(md_content)
        
        print(f"✓ Report generated: {output_file}")
        print(f"  - {len(results)} pages documented")
        
    except Exception as e:
        print(f"✗ Error generating report: {e}")
        import traceback
        traceback.print_exc()

# Generate the report
generate_confluence_report(confluence, output_file="confluence_pages_report.md")


Searching Confluence for pages...
Found 8 pages, generating report...
✓ Report generated: confluence_pages_report.md
  - 8 pages documented


## Generate Confluence Pages Report

In [ ]:
# Get page content
page_id = "12345"  # Replace with actual page ID
page_content = confluence.get_page_by_id(page_id, expand="body.storage")
print("Page content:")
print(page_content['body']['storage']['value'])

In [ ]:
# Get a specific page by key
page = confluence.get_page_by_title(space_key="SPACE_KEY", title="Page Title")
print(f"Page ID: {page['id']}")
print(f"Page Title: {page['title']}")

## Common Operations

In [11]:
import subprocess
import sys
import os

# Install required library
subprocess.check_call([sys.executable, "-m", "pip", "install", "atlassian-python-api", "-q"])

from atlassian import Confluence
import time

# Disable SSL warnings (optional - for development only)
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Connection parameters
confluence_url = "https://apimigration.atlassian.net"
username = "mangesh.joshi@coforge.com"
api_token = "ATATT3xFfGF0FBeNKWzSFzmXm433k-QLw5-G3WFSjpBU1xGwtcpoSmERzX2Hqhb2s3EZhhETKHuYWZuxuWb9xUmCLNdKg3aU15S48lHde5PPrsFj1QDdC1nVPQPwpYV72UjnQ0G3Bag5PgE0McETNiqng3FYW7psoQL2LBLHjworggAoLKnW_fg=53B7DC84"

print("Establishing Confluence connection...")
start_time = time.time()

try:
    # Connect with SSL verification disabled (for troubleshooting)
    # In production, use proper certificate handling
    confluence = Confluence(
        url=confluence_url,
        username=username,
        password=api_token,
        cloud=True,
        verify_ssl=False  # Disable SSL verification
    )
    
    # Test the connection by getting spaces
    spaces = confluence.get_all_spaces()
    elapsed = time.time() - start_time
    print(f"✓ Connected successfully!")
    print(f"  Connection established in {elapsed:.2f} seconds")
    print(f"  Found {len(spaces['results']) if 'results' in spaces else 0} spaces")
    
except Exception as e:
    elapsed = time.time() - start_time
    print(f"✗ Connection failed after {elapsed:.2f} seconds")
    print(f"Error: {str(e)}")
    print("\nTroubleshooting:")
    print("  1. Verify confluence_url is correct")
    print("  2. Verify username is correct")
    print("  3. Verify API token is valid (generate new one if expired)")
    print("  4. Check if token has required permissions")
    print("  5. If SSL error persists, check proxy settings or firewall")
    print("\nSSL Error Help:")
    print("  - verify_ssl=False is currently enabled for troubleshooting")
    print("  - In production, use proper SSL certificates")


Establishing Confluence connection...
✓ Connected successfully!
  Connection established in 0.75 seconds
  Found 2 spaces


In [ ]:
# Install required library
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "atlassian-python-api", "-q"])

# Confluence Connection Guide

This notebook has functions to connect to Confluence and run cql to extract pages and generate mark up file.  This will be used as one of the inputs to generate technical specifications for forward engineering of api code.